# High-price valuation validation

Issue #18. Use training and 2024 validation only; never inspect or score 2025 test sales here. Outputs are aggregates, with no sale-level records. Run from the repo root after preparing the cohort.

Decision rule: change the fixed loss only if high-price MAE falls at least 10% and overall validation MAE rises no more than 5%. The price threshold is the training-set p90. Intervals are diagnostic, not a serving contract.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from homelens.modeling.baseline import CATEGORICAL_COLUMNS, MODEL_CONFIG, _features
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

cohort = pd.read_csv(
    Path("data/processed/modeling_cohort.csv"), dtype={"zip": "string"}
)
cohort = cohort.loc[cohort["split"].isin(["train", "validation"])].copy()
assert set(cohort["split"]) == {"train", "validation"}
train = cohort.loc[cohort["split"].eq("train")].copy()
validation = cohort.loc[cohort["split"].eq("validation")].copy()
assert train["sale_date"].max() < validation["sale_date"].min()
p90 = float(train["price_usd"].quantile(0.9))
high = validation["price_usd"].gt(p90).to_numpy()
zip_high = (
    validation["zip"].eq("27707")
    & validation["property_type"].eq("Single Family Residential")
).to_numpy() & high
print(
    {
        "train_rows": len(train),
        "validation_rows": len(validation),
        "training_p90_usd": round(p90),
        "validation_high_price_rows": int(high.sum()),
        "zip_27707_single_family_high_rows": int(zip_high.sum()),
    }
)

{'train_rows': 13280, 'validation_rows': 2979, 'training_p90_usd': 600000, 'validation_high_price_rows': 420, 'zip_27707_single_family_high_rows': 118}


## Fixed-loss comparison

Only the loss changes. Signed bias is prediction minus sale price; negative means underprediction.

In [2]:
categories = {
    column: sorted(train[column].astype("string").unique().tolist())
    for column in CATEGORICAL_COLUMNS
}
train_features = _features(train, categories)
validation_features = _features(validation, categories)
truth = validation["price_usd"].to_numpy(dtype=float)
predictions, rows = {}, []
for loss in ("absolute_error", "squared_error", "poisson"):
    model = HistGradientBoostingRegressor(**{**MODEL_CONFIG, "loss": loss})
    model.fit(train_features, train["price_usd"])
    predicted = model.predict(validation_features)
    predictions[loss] = predicted
    rows.append(
        {
            "loss": loss,
            "overall_mae_usd": round(mean_absolute_error(truth, predicted)),
            "overall_rmse_usd": round(root_mean_squared_error(truth, predicted)),
            "high_price_mae_usd": round(
                mean_absolute_error(truth[high], predicted[high])
            ),
            "high_price_bias_usd": round(float(np.mean(predicted[high] - truth[high]))),
            "zip_27707_single_family_high_mae_usd": round(
                mean_absolute_error(truth[zip_high], predicted[zip_high])
            ),
        }
    )
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
reference = comparison.iloc[0]
eligible = comparison.loc[
    (comparison["high_price_mae_usd"] <= reference["high_price_mae_usd"] * 0.9)
    & (comparison["overall_mae_usd"] <= reference["overall_mae_usd"] * 1.05)
]
chosen_loss = (
    str(eligible.sort_values("overall_mae_usd").iloc[0]["loss"])
    if not eligible.empty
    else "absolute_error"
)
print("validation_chosen_loss:", chosen_loss)

          loss  overall_mae_usd  overall_rmse_usd  high_price_mae_usd  high_price_bias_usd  zip_27707_single_family_high_mae_usd
absolute_error            78569            159868              250076              -233579                                418392
 squared_error            74807            138001              213299              -187525                                334548
       poisson            74463            138199              212102              -186127                                333654
validation_chosen_loss: poisson


## Temporal interval check

Calibrate a symmetric 90% absolute-residual interval on January-June 2024, then assess July-December 2024. Check whether one global width covers expensive homes.

In [3]:
calibration = validation["sale_date"].lt("2024-07-01").to_numpy()
assessment = ~calibration
interval_rows = []
for loss in dict.fromkeys(("absolute_error", chosen_loss)):
    predicted = predictions[loss]
    errors = np.abs(truth[calibration] - predicted[calibration])
    level = min(1.0, np.ceil((len(errors) + 1) * 0.9) / len(errors))
    width = float(np.quantile(errors, level, method="higher"))
    covered = np.abs(truth - predicted) <= width
    high_assessment = assessment & high
    interval_rows.append(
        {
            "loss": loss,
            "calibration_rows": int(calibration.sum()),
            "assessment_rows": int(assessment.sum()),
            "high_price_assessment_rows": int(high_assessment.sum()),
            "half_width_usd": round(width),
            "assessment_coverage": round(float(covered[assessment].mean()), 3),
            "high_price_coverage": round(float(covered[high_assessment].mean()), 3),
        }
    )
print(pd.DataFrame(interval_rows).to_string(index=False))

          loss  calibration_rows  assessment_rows  high_price_assessment_rows  half_width_usd  assessment_coverage  high_price_coverage
absolute_error              1548             1431                         215          156896                0.898                0.470
       poisson              1548             1431                         215          151142                0.896                0.493


## Interpretation

The Poisson loss meets the prespecified rule: overall validation MAE falls from USD 78,569 to USD 74,463, and high-price MAE falls from USD 250,076 to USD 212,102 (about 15%). The high-price signed bias remains negative (about USD 186,127), so the tail problem is improved, not solved. The 118 high-price single-family validation sales in 27707 also improve from USD 418,392 to USD 333,654 MAE.

A single symmetric 90% interval calibrated on early 2024 covers only 49.3% of high-price sales in late 2024 for the chosen model, despite 89.6% overall coverage. Do not ship this interval. Translate the fixed Poisson comparison into tested Python code, then run the 2025 split once for a final descriptive check. No county-wide claim is justified until issue #12 is resolved.